# Design Problem

# GP solutions

##  Problem definition

After inspecting the contents of `disc-benchmark-files`, the system-identification task can be reduced to two closely related subproblems: *one-step-ahead prediction* and *free-run simulation* of the unbalanced disk angle. The available measured signals are the applied input voltage `u` and the measured disk angle `th`. The aim is therefore to learn a model that can estimate the next angle sample from delayed input and output sequences.

As in the ANN solution, a *NARX structure* is used. The model receives past input samples and past output samples as one regressor vector and predicts the next output sample. The GP version differs from the ANN version because it uses a Gaussian Process regression model, which gives both a mean prediction and an uncertainty estimate.

The notebook  implements the GP-NARX model training and validation in the following sequence: 
1.  helper functions definition
2. preparation of the benchmark data
3. training several GP-NARX models
4. saving each trained model
5. evaluation of one-step prediction and free-run simulation performance
6. comparison of the models by tables and visualisations.



## 1.2 Repository paths

The benchmark data are stored in `gym-unbalanced-disk/disc-benchmark-files`, while all GP outputs are written into `SystemModelling`.

In [1]:
from pathlib import Path
import time
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore", category=UserWarning)

def find_project_root(start=None):

    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    candidates = [start, start.parent, *start.parents]
    for candidate in candidates:
        benchmark_dir = candidate / "gym-unbalanced-disk" / "disc-benchmark-files"
        if benchmark_dir.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find project root. Set PROJECT_ROOT manually, e.g. "
        "PROJECT_ROOT = Path('/path/to/DesignProject').resolve()"
    )

PROJECT_ROOT = find_project_root()
BENCHMARK_DIR = PROJECT_ROOT / "gym-unbalanced-disk" / "disc-benchmark-files"
SYSTEM_MODELLING_DIR = PROJECT_ROOT / "SystemModelling"

RUNS_DIR = SYSTEM_MODELLING_DIR / "gp_narx_runs_combined"
VIS_DIR = SYSTEM_MODELLING_DIR / "gp_narx_visualizations_combined"

RUNS_DIR.mkdir(parents=True, exist_ok=True)
VIS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DATA_PATH = BENCHMARK_DIR / "training-val-test-data.npz"
PRED_HIDDEN_PATH = BENCHMARK_DIR / "hidden-test-prediction-submission-file.npz"
SIM_HIDDEN_PATH = BENCHMARK_DIR / "hidden-test-simulation-submission-file.npz"



## 2 Gaussian Process NARX solution

### 2.1 Approach

A NARX structure was selected because the dynamics of the unbalanced disk does not depend only on the current voltage input it also depends on the recent  measured angles and on previous voltage values. Past angle memory is described as (`na`) and  past voltage memory as (`nb`). For a chosen order `(na, nb)`, the regressor is built as:

```text
X[k] = [u[k-nb], ..., u[k-1], th[k-na], ..., th[k-1]]
Y[k] = th[k]
```

The Gaussian Process is trained on these regressors and the model is then evaluated in two ways:
 First, in *one-step prediction*, the GP always receives the true measured past outputs. 
 Second, in *free-run simulation*, the GP uses its own previous predictions recursively and this case also more important for analysing long-term simulation behaviour.

### 2.2 Workflow

The notebook follows these steps:

1. load the provided benchmark data;
2. split the time series without shuffling;
3. build NARX input-output matrices;
4. train a GP with an ARD-RBF kernel;
5. save the trained GP as a `.joblib` model ;
6. evaluate one-step prediction and free-run simulation;
7. run additional asymmetric order tests;
8. visualise the best model and compare all tested orders.


## 2. Core GP-NARX functions



In [2]:
def create_io_data(u, th, na, nb):
    """

    X[k] = [u[k-nb], ..., u[k-1], th[k-na], ..., th[k-1]]
    Y[k] = th[k]
    """
    u = np.asarray(u).reshape(-1)
    th = np.asarray(th).reshape(-1)

    if len(u) != len(th):
        raise ValueError("u and th must have the same length.")

    X = []
    Y = []

    start = max(na, nb)
    for k in range(start, len(th)):
        xk = np.concatenate([u[k - nb:k], th[k - na:k]])
        X.append(xk)
        Y.append(th[k])

    return np.asarray(X), np.asarray(Y)


def time_series_split(u, th, val_fraction=0.2):
    """
    Split a time series without shuffling.
    """
    n = len(th)
    n_val = int(n * val_fraction)
    n_train = n - n_val
    return u[:n_train], th[:n_train], u[n_train:], th[n_train:]


def choose_training_subset(X, Y, max_train_samples=2000, seed=42):
    """
    Select a random subset for GP hyperparameter training.
    """
    if len(X) <= max_train_samples:
        return X, Y

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_train_samples, replace=False)
    idx = np.sort(idx)
    return X[idx], Y[idx]


def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    nrms = rmse / np.std(y_true) * 100.0

    return {
        "rmse_rad": float(rmse),
        "rmse_deg": float(np.rad2deg(rmse)),
        "mae_rad": float(mae),
        "nrms_percent": float(nrms),
    }


def print_metrics(name, y_true, y_pred):
    m = compute_metrics(y_true, y_pred)

    print(f"\n{name}")
    print(f"RMSE : {m['rmse_rad']:.6f} rad")
    print(f"RMSE : {m['rmse_deg']:.6f} deg")
    print(f"MAE  : {m['mae_rad']:.6f} rad")
    print(f"NRMS : {m['nrms_percent']:.2f} %")

    return m


def plot_prediction(path, y_true, y_pred, std=None, title=""):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(12, 4))
    plt.plot(y_true, label="Measured")
    plt.plot(y_pred, label="GP")

    if std is not None:
        x = np.arange(len(y_pred))
        plt.fill_between(
            x,
            y_pred - 2.0 * std,
            y_pred + 2.0 * std,
            alpha=0.2,
            label="GP ±2 std",
        )

    plt.xlabel("Sample")
    plt.ylabel("Angle th [rad]")
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print("Saved:", path)

In [3]:
class GPNARX:
    """
    GP-NARX model.

    Feature ordering:
        [u[k-nb], ..., u[k-1], th[k-na], ..., th[k-1]]
    """

    def __init__(self, na=5, nb=5, max_train_samples=2000, seed=42, n_restarts_optimizer=3):
        self.na = int(na)
        self.nb = int(nb)
        self.max_train_samples = int(max_train_samples)
        self.seed = int(seed)
        self.n_restarts_optimizer = int(n_restarts_optimizer)

        self.x_scaler = StandardScaler()
        self.y_scaler = StandardScaler()
        self.gp = None
        self.training_time_s = None

    def fit(self, u, th):
        X, Y = create_io_data(u, th, self.na, self.nb)
        X_fit, Y_fit = choose_training_subset(
            X,
            Y,
            max_train_samples=self.max_train_samples,
            seed=self.seed,
        )

        X_fit_s = self.x_scaler.fit_transform(X_fit)
        Y_fit_s = self.y_scaler.fit_transform(Y_fit.reshape(-1, 1)).ravel()

        n_features = X_fit_s.shape[1]

        kernel = (
            ConstantKernel(1.0, (1e-2, 1e2))
            * RBF(
                length_scale=np.ones(n_features),
                length_scale_bounds=(1e-2, 1e2),
            )
            + WhiteKernel(
                noise_level=1e-3,
                noise_level_bounds=(1e-7, 1e0),
            )
        )

        self.gp = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=False,
            n_restarts_optimizer=self.n_restarts_optimizer,
            random_state=self.seed,
        )

        print(f"\nTraining GP-NARX with na={self.na}, nb={self.nb}")
        print(f"Total NARX samples: {len(X)}")
        print(f"Samples used for GP training: {len(X_fit)}")

        t0 = time.time()
        self.gp.fit(X_fit_s, Y_fit_s)
        self.training_time_s = time.time() - t0

        print(f"Training time: {self.training_time_s:.1f} s")
        print("Optimised kernel:")
        print(self.gp.kernel_)

        return self

    def predict_from_x(self, x, return_std=False):
        """
        Predict from one or more NARX regressors.
        """
        x = np.asarray(x)

        if x.ndim == 1:
            x = x[None, :]

        xs = self.x_scaler.transform(x)

        if return_std:
            yp_s, std_s = self.gp.predict(xs, return_std=True)
            yp = self.y_scaler.inverse_transform(yp_s.reshape(-1, 1)).ravel()
            std = std_s * self.y_scaler.scale_[0]
            return yp, std

        yp_s = self.gp.predict(xs)
        yp = self.y_scaler.inverse_transform(yp_s.reshape(-1, 1)).ravel()
        return yp

    def one_step_prediction(self, u, th, return_std=False):
        """
        One-step prediction using measured past th.
        """
        X, Y = create_io_data(u, th, self.na, self.nb)

        if return_std:
            Yp, std = self.predict_from_x(X, return_std=True)
            return Y, Yp, std

        Yp = self.predict_from_x(X, return_std=False)
        return Y, Yp

    def hidden_prediction(self, upast, thpast):
        """
        Prediction for hidden-test-prediction-submission-file.npz.

        Provided arrays:
            upast:  N x 15, columns [u[k-15], ..., u[k-1]]
            thpast: N x 15, columns [th[k-15], ..., th[k-1]]
        """
        X = np.concatenate(
            [upast[:, 15 - self.nb:], thpast[:, 15 - self.na:]],
            axis=1,
        )
        return self.predict_from_x(X)

    def simulate(self, u, th_initial, skip=50):
        """
        Free-run simulation using the model's own previous angle predictions.
        """
        u = np.asarray(u).reshape(-1)
        th_initial = np.asarray(th_initial).reshape(-1)

        if skip < max(self.na, self.nb):
            raise ValueError("skip must be at least max(na, nb).")

        if len(th_initial) < skip:
            raise ValueError("th_initial must contain at least skip samples.")

        Y = th_initial[:skip].astype(float).tolist()
        upast = u[skip - self.nb:skip].astype(float).tolist()
        thpast = th_initial[skip - self.na:skip].astype(float).tolist()

        for uk in u[skip:]:
            x = np.concatenate([upast, thpast])
            ypred = float(self.predict_from_x(x)[0])

            Y.append(ypred)

            upast.append(float(uk))
            upast.pop(0)

            thpast.append(ypred)
            thpast.pop(0)

        return np.asarray(Y)

    def to_bundle(self):
        return {
            "gp": self.gp,
            "x_scaler": self.x_scaler,
            "y_scaler": self.y_scaler,
            "na": self.na,
            "nb": self.nb,
            "max_train_samples": self.max_train_samples,
            "seed": self.seed,
            "n_restarts_optimizer": self.n_restarts_optimizer,
            "training_time_s": self.training_time_s,
            "kernel": str(self.gp.kernel_),
        }

## 3. Load becnhmark data


```text
gym-unbalanced-disk/disc-benchmark-files/training-val-test-data.npz
```

In [4]:
train_data = np.load(TRAIN_DATA_PATH)

u_all = np.asarray(train_data["u"]).reshape(-1)
th_all = np.asarray(train_data["th"]).reshape(-1)

# Assignment voltage range.
u_all = np.clip(u_all, -3.0, 3.0)

VAL_FRACTION = 0.2

u_train, th_train, u_val, th_val = time_series_split(
    u_all,
    th_all,
    val_fraction=VAL_FRACTION,
)

print("Loaded training data")
print("u_all shape:", u_all.shape)
print("th_all shape:", th_all.shape)
print("u_train:", u_train.shape, "th_train:", th_train.shape)
print("u_val:", u_val.shape, "th_val:", th_val.shape)
print("u range:", (float(u_all.min()), float(u_all.max())))
print("th range:", (float(th_all.min()), float(th_all.max())))

Loaded training data
u_all shape: (35000,)
th_all shape: (35000,)
u_train: (28000,) th_train: (28000,)
u_val: (7000,) th_val: (7000,)
u range: (-3.0, 3.0)
th range: (-1.900663555421825, 2.1802653015913167)


## 4. Train and evaluate one GP-NARX model

This function trains one GP-NARX model, saves the model bundle, saves one-step and free-run plots, creates hidden-test submission files, and returns a metrics dictionary.

In [6]:
def train_evaluate_gp_order(
    na,
    nb,
    max_train_samples=2000,
    seed=42,
    n_restarts_optimizer=3,
    runs_dir=RUNS_DIR,
    make_hidden_submissions=True,
):
    """
    Train and evaluate one GP-NARX models.
    """
    run_name = f"na{na}_nb{nb}"
    out_dir = Path(runs_dir) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)

    model = GPNARX(
        na=na,
        nb=nb,
        max_train_samples=max_train_samples,
        seed=seed,
        n_restarts_optimizer=n_restarts_optimizer,
    )

    model.fit(u_train, th_train)


    model_path = out_dir / f"gp_narx_bundle_na{na}_nb{nb}.joblib"
    joblib.dump(model.to_bundle(), model_path)
    print("\nSaved GP model bundle:", model_path)

    # One-step prediction.
    y_true_pred, y_pred, y_std = model.one_step_prediction(
        u_val,
        th_val,
        return_std=True,
    )

    pred_metrics = print_metrics("Validation one-step prediction", y_true_pred, y_pred)

    plot_prediction(
        out_dir / "validation_one_step_prediction.png",
        y_true_pred,
        y_pred,
        std=y_std,
        title=f"GP-NARX one-step prediction, na={na}, nb={nb}",
    )

    # Free-run simulation.
    skip_val = max(na, nb)
    th_sim_val = model.simulate(
        u_val,
        th_initial=th_val,
        skip=skip_val,
    )

    sim_metrics = print_metrics(
        "Validation free-run simulation",
        th_val[skip_val:],
        th_sim_val[skip_val:],
    )

    plot_prediction(
        out_dir / "validation_free_run_simulation.png",
        th_val[skip_val:],
        th_sim_val[skip_val:],
        title=f"GP-NARX free-run simulation, na={na}, nb={nb}",
    )

    # Hidden-test submission files.
    pred_submission_path = None
    sim_submission_path = None

    if make_hidden_submissions:
        pred_data = np.load(PRED_HIDDEN_PATH)
        upast_test = pred_data["upast"]
        thpast_test = pred_data["thpast"]

        thnow_pred = model.hidden_prediction(upast_test, thpast_test)

        pred_submission_path = out_dir / "gp_hidden_prediction_submission.npz"
        np.savez(
            pred_submission_path,
            upast=upast_test,
            thpast=thpast_test,
            thnow=thnow_pred,
        )

        sim_data = np.load(SIM_HIDDEN_PATH)
        u_test = np.clip(sim_data["u"], -3.0, 3.0)
        th_test_template = sim_data["th"]

        skip_hidden = 50
        th_hidden_sim = model.simulate(
            u_test,
            th_initial=th_test_template,
            skip=skip_hidden,
        )

        th_hidden_sim[:skip_hidden] = th_test_template[:skip_hidden]

        sim_submission_path = out_dir / "gp_hidden_simulation_submission.npz"
        np.savez(
            sim_submission_path,
            th=th_hidden_sim,
            u=u_test,
        )

        print("\nSaved hidden prediction submission:", pred_submission_path)
        print("Saved hidden simulation submission:", sim_submission_path)

    result = {
        "order": f"({na},{nb})",
        "na": int(na),
        "nb": int(nb),
        "max_train_samples": int(max_train_samples),
        "seed": int(seed),
        "n_restarts_optimizer": int(n_restarts_optimizer),
        "training_time_s": float(model.training_time_s),
        "pred_rmse_rad": pred_metrics["rmse_rad"],
        "pred_rmse_deg": pred_metrics["rmse_deg"],
        "pred_mae_rad": pred_metrics["mae_rad"],
        "pred_nrms_percent": pred_metrics["nrms_percent"],
        "sim_rmse_rad": sim_metrics["rmse_rad"],
        "sim_rmse_deg": sim_metrics["rmse_deg"],
        "sim_mae_rad": sim_metrics["mae_rad"],
        "sim_nrms_percent": sim_metrics["nrms_percent"],
        "kernel": str(model.gp.kernel_),
        "model_path": str(model_path),
        "run_dir": str(out_dir),
        "pred_submission_path": None if pred_submission_path is None else str(pred_submission_path),
        "sim_submission_path": None if sim_submission_path is None else str(sim_submission_path),
    }

    with open(out_dir / "metrics.json", "w") as f:
        json.dump(result, f, indent=2)

    return result

## 5. Define the GP order tests

The first four are the symmetric tests.  
The next four are the extra asymmetric tests to  check whether past angle memory (`na`) or past voltage memory (`nb`) matters more.

In [7]:
BASE_ORDERS = [
    (5, 5),
    (6, 6),
    (7, 7),
    (10, 10),
]

EXTRA_ORDERS = [
    (10, 5),
    (5, 10),
    (10, 7),
    (7, 10),
]

ORDERS_TO_RUN = BASE_ORDERS + EXTRA_ORDERS

MAX_TRAIN_SAMPLES = 2000
SEED = 21
N_RESTARTS_OPTIMIZER = 3

print("Orders to run:")
for na, nb in ORDERS_TO_RUN:
    print(f"  na={na}, nb={nb}")

Orders to run:
  na=5, nb=5
  na=6, nb=6
  na=7, nb=7
  na=10, nb=10
  na=10, nb=5
  na=5, nb=10
  na=10, nb=7
  na=7, nb=10


## 6. Run all GP tests


In [8]:
all_results = []

for na, nb in ORDERS_TO_RUN:
    result = train_evaluate_gp_order(
        na=na,
        nb=nb,
        max_train_samples=MAX_TRAIN_SAMPLES,
        seed=SEED,
        n_restarts_optimizer=N_RESTARTS_OPTIMIZER,
        runs_dir=RUNS_DIR,
        make_hidden_submissions=True,
    )
    all_results.append(result)

results_df = pd.DataFrame(all_results)

results_csv_path = RUNS_DIR / "gp_order_comparison.csv"
results_df.to_csv(results_csv_path, index=False)

print("Saved comparison table:", results_csv_path)

display(
    results_df[
        [
            "order",
            "training_time_s",
            "pred_rmse_rad",
            "pred_rmse_deg",
            "pred_nrms_percent",
            "sim_rmse_rad",
            "sim_rmse_deg",
            "sim_nrms_percent",
        ]
    ].sort_values("sim_rmse_rad")
)


Training GP-NARX with na=5, nb=5
Total NARX samples: 27995
Samples used for GP training: 2000


KeyboardInterrupt: 

## 7. Analyse and visualise the saved results.

In [9]:
results_csv_path = RUNS_DIR / "gp_order_comparison.csv"

if results_csv_path.exists():
    results_df = pd.read_csv(results_csv_path)
    print("Loaded:", results_csv_path)
    display(
        results_df[
            [
                "order",
                "training_time_s",
                "pred_rmse_rad",
                "pred_rmse_deg",
                "pred_nrms_percent",
                "sim_rmse_rad",
                "sim_rmse_deg",
                "sim_nrms_percent",
            ]
        ].sort_values("sim_rmse_rad")
    )
else:
    print("No saved comparison table found yet. Run the training cell first.")

No saved comparison table found yet. Run the training cell first.


## 8. Plot the order comparison

These plots are used to show:

- one-step prediction error;
- free-run simulation error;
- training time.

In [10]:
if "results_df" not in globals() or results_df.empty:
    raise RuntimeError("results_df is missing. Run the training cell or load the CSV first.")

plot_df = results_df.sort_values("sim_rmse_rad").copy()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["order"], plot_df["pred_rmse_rad"])
plt.xlabel("GP-NARX order (na, nb)")
plt.ylabel("One-step prediction RMSE [rad]")
plt.title("GP-NARX one-step prediction RMSE by order")
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(RUNS_DIR / "comparison_prediction_rmse.png", dpi=200)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["order"], plot_df["sim_rmse_rad"])
plt.xlabel("GP-NARX order (na, nb)")
plt.ylabel("Free-run simulation RMSE [rad]")
plt.title("GP-NARX free-run simulation RMSE by order")
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(RUNS_DIR / "comparison_simulation_rmse.png", dpi=200)
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(plot_df["order"], plot_df["training_time_s"])
plt.xlabel("GP-NARX order (na, nb)")
plt.ylabel("Training time [s]")
plt.title("GP-NARX training time by order")
plt.grid(True, axis="y")
plt.tight_layout()
plt.savefig(RUNS_DIR / "comparison_training_time.png", dpi=200)
plt.show()

print("Saved comparison plots in:", RUNS_DIR)

RuntimeError: results_df is missing. Run the training cell or load the CSV first.

## 9. Select the best model

select the model with the lowest *free-run simulation RMSE*, because free-run simulation is harder than one-step prediction and better reflects long-term model quality.

In [10]:
best_row = results_df.sort_values("sim_rmse_rad").iloc[0]
BEST_MODEL_PATH = Path(best_row["model_path"])
BEST_RUN_DIR = Path(best_row["run_dir"])

print("Best model by free-run simulation RMSE:")
display(best_row.to_frame().T)

print("BEST_MODEL_PATH:", BEST_MODEL_PATH)
print("BEST_RUN_DIR:", BEST_RUN_DIR)

Best model by free-run simulation RMSE:


,order,na,nb,max_train_samples,seed,n_restarts_optimizer,training_time_s,pred_rmse_rad,pred_rmse_deg,pred_mae_rad,pred_nrms_percent,sim_rmse_rad,sim_rmse_deg,sim_mae_rad,sim_nrms_percent,kernel,model_path,run_dir,pred_submission_path,sim_submission_path
0,"(5,5)",5,5,2000,21,3,200.397251,0.003541,0.202902,0.00252,0.68476,0.036455,2.088694,0.024528,7.048989,"4.62**2 * RBF(length_scale=[100, 100, 100, 100...",/gpfs/home4/scur2879/DesignProject/SystemModel...,/gpfs/home4/scur2879/DesignProject/SystemModel...,/gpfs/home4/scur2879/DesignProject/SystemModel...,/gpfs/home4/scur2879/DesignProject/SystemModel...


BEST_MODEL_PATH: /gpfs/home4/scur2879/DesignProject/SystemModelling/gp_narx_runs_combined/na5_nb5/gp_narx_bundle_na5_nb5.joblib
BEST_RUN_DIR: /gpfs/home4/scur2879/DesignProject/SystemModelling/gp_narx_runs_combined/na5_nb5


## 10. Visualisation functions

This section contains the visualisation code.

In [11]:
def load_gp_model(model_path):

    model_path = Path(model_path).resolve()
    obj = joblib.load(model_path)

    if isinstance(obj, dict):
        required = ["gp", "x_scaler", "y_scaler", "na", "nb"]
        missing = [k for k in required if k not in obj]
        if missing:
            raise ValueError(f"Model bundle is missing keys: {missing}")

        return {
            "gp": obj["gp"],
            "x_scaler": obj["x_scaler"],
            "y_scaler": obj["y_scaler"],
            "na": int(obj["na"]),
            "nb": int(obj["nb"]),
            "kernel": obj.get("kernel", str(obj["gp"].kernel_)),
        }

    required_attrs = ["gp", "x_scaler", "y_scaler", "na", "nb"]
    missing = [a for a in required_attrs if not hasattr(obj, a)]
    if missing:
        raise ValueError(
            f"Loaded object is not a valid GP-NARX model. Missing: {missing}"
        )

    return {
        "gp": obj.gp,
        "x_scaler": obj.x_scaler,
        "y_scaler": obj.y_scaler,
        "na": int(obj.na),
        "nb": int(obj.nb),
        "kernel": str(obj.gp.kernel_),
    }


def predict_from_x_loaded(model, X, return_std=False):
    X = np.asarray(X)

    if X.ndim == 1:
        X = X[None, :]

    Xs = model["x_scaler"].transform(X)

    if return_std:
        yp_s, std_s = model["gp"].predict(Xs, return_std=True)
        yp = model["y_scaler"].inverse_transform(yp_s.reshape(-1, 1)).reshape(-1)
        std = std_s * model["y_scaler"].scale_[0]
        return yp, std

    yp_s = model["gp"].predict(Xs)
    yp = model["y_scaler"].inverse_transform(yp_s.reshape(-1, 1)).reshape(-1)
    return yp


def one_step_prediction_loaded(model, u, th, return_std=True):
    na = model["na"]
    nb = model["nb"]
    X, Y = create_io_data(u, th, na, nb)

    if return_std:
        Yp, std = predict_from_x_loaded(model, X, return_std=True)
        return X, Y, Yp, std

    Yp = predict_from_x_loaded(model, X, return_std=False)
    return X, Y, Yp


def simulate_free_run_loaded(model, u, th_initial, skip):
    na = model["na"]
    nb = model["nb"]

    u = np.asarray(u).reshape(-1)
    th_initial = np.asarray(th_initial).reshape(-1)

    if skip < max(na, nb):
        raise ValueError("skip must be at least max(na, nb).")

    if len(th_initial) < skip:
        raise ValueError("th_initial must contain at least skip samples.")

    Y = th_initial[:skip].astype(float).tolist()
    upast = u[skip - nb:skip].astype(float).tolist()
    thpast = th_initial[skip - na:skip].astype(float).tolist()

    for uk in u[skip:]:
        x = np.concatenate([upast, thpast])
        ypred = float(predict_from_x_loaded(model, x)[0])

        Y.append(ypred)

        upast.append(float(uk))
        upast.pop(0)

        thpast.append(ypred)
        thpast.pop(0)

    return np.asarray(Y)


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close()
    print("Saved:", path)


def get_feature_labels(na, nb):
    labels = []

    for lag in range(nb, 0, -1):
        labels.append(f"u[k-{lag}]")

    for lag in range(na, 0, -1):
        labels.append(f"th[k-{lag}]")

    return labels

In [12]:
def plot_one_step_with_uncertainty(out_dir, y_true, y_pred, std):
    x = np.arange(len(y_true))

    plt.figure(figsize=(13, 5))
    plt.plot(x, y_true, label="Measured")
    plt.plot(x, y_pred, label="GP mean prediction")
    plt.fill_between(
        x,
        y_pred - 2.0 * std,
        y_pred + 2.0 * std,
        alpha=0.25,
        label="GP ±2 standard deviations",
    )

    plt.xlabel("Validation sample")
    plt.ylabel("Angle th [rad]")
    plt.title("One-step GP-NARX prediction with uncertainty")
    plt.grid(True)
    plt.legend()

    savefig(Path(out_dir) / "01_one_step_prediction_uncertainty.png")


def plot_free_run_simulation(out_dir, th_val, th_sim, skip):
    x = np.arange(skip, len(th_val))

    plt.figure(figsize=(13, 5))
    plt.plot(x, th_val[skip:], label="Measured")
    plt.plot(x, th_sim[skip:], label="GP free-run simulation")

    plt.xlabel("Validation sample")
    plt.ylabel("Angle th [rad]")
    plt.title("Free-run GP-NARX simulation")
    plt.grid(True)
    plt.legend()

    savefig(Path(out_dir) / "02_free_run_simulation.png")


def plot_residual_time_series(out_dir, y_true, y_pred, name="one_step"):
    residual = y_true - y_pred

    plt.figure(figsize=(13, 4))
    plt.plot(residual)

    plt.xlabel("Sample")
    plt.ylabel("Residual [rad]")
    plt.title(f"Residual over time: {name}")
    plt.grid(True)

    savefig(Path(out_dir) / f"03_residual_time_series_{name}.png")


def plot_residual_histogram(out_dir, y_true, y_pred, name="one_step"):
    residual = y_true - y_pred

    plt.figure(figsize=(8, 5))
    plt.hist(residual, bins=60, alpha=0.85)

    plt.xlabel("Residual [rad]")
    plt.ylabel("Count")
    plt.title(f"Residual histogram: {name}")
    plt.grid(True)

    savefig(Path(out_dir) / f"04_residual_histogram_{name}.png")


def plot_predictive_std(out_dir, std):
    plt.figure(figsize=(13, 4))
    plt.plot(std)

    plt.xlabel("Validation sample")
    plt.ylabel("Predictive standard deviation [rad]")
    plt.title("GP predictive uncertainty over time")
    plt.grid(True)

    savefig(Path(out_dir) / "05_predictive_std_over_time.png")


def plot_kernel_lengthscales(out_dir, model):
    gp = model["gp"]
    na = model["na"]
    nb = model["nb"]

    labels = get_feature_labels(na, nb)

    try:
        length_scales = gp.kernel_.k1.k2.length_scale
    except Exception as e:
        print("Could not extract length-scales from kernel.")
        print("Kernel was:", gp.kernel_)
        print("Error:", e)
        return

    length_scales = np.asarray(length_scales).reshape(-1)

    plt.figure(figsize=(12, 5))
    plt.bar(labels, length_scales)

    plt.xlabel("NARX regressor")
    plt.ylabel("ARD length-scale")
    plt.title("Optimised GP RBF length-scales")
    plt.xticks(rotation=45, ha="right")
    plt.grid(True, axis="y")

    savefig(Path(out_dir) / "06_kernel_lengthscales.png")

In [13]:
def plot_gp_slice_1d(
    out_dir,
    model,
    x_ref,
    dim_to_vary,
    x_min,
    x_max,
    n_points=250,
    n_func_samples=8,
):
    labels = get_feature_labels(model["na"], model["nb"])
    dim_label = labels[dim_to_vary]

    x_ref = np.asarray(x_ref).reshape(-1)
    X_plot = np.tile(x_ref, (n_points, 1))

    x_values = np.linspace(x_min, x_max, n_points)
    X_plot[:, dim_to_vary] = x_values

    X_plot_s = model["x_scaler"].transform(X_plot)

    mean_s, std_s = model["gp"].predict(X_plot_s, return_std=True)
    mean = model["y_scaler"].inverse_transform(mean_s.reshape(-1, 1)).reshape(-1)
    std = std_s * model["y_scaler"].scale_[0]

    samples_s = model["gp"].sample_y(
        X_plot_s,
        n_samples=n_func_samples,
        random_state=0,
    )

    samples = np.zeros_like(samples_s)
    for i in range(n_func_samples):
        samples[:, i] = model["y_scaler"].inverse_transform(
            samples_s[:, i].reshape(-1, 1)
        ).reshape(-1)

    plt.figure(figsize=(10, 6))
    plt.plot(x_values, mean, label="GP posterior mean")
    plt.fill_between(
        x_values,
        mean - 2.0 * std,
        mean + 2.0 * std,
        alpha=0.25,
        label="±2 standard deviations",
    )

    for i in range(n_func_samples):
        plt.plot(
            x_values,
            samples[:, i],
            linestyle="--",
            linewidth=1,
            alpha=0.75,
        )

    plt.xlabel(dim_label)
    plt.ylabel("Predicted th[k] [rad]")
    plt.title(f"1D GP posterior slice varying {dim_label}")
    plt.grid(True)
    plt.legend()

    safe_dim_label = dim_label.replace("[", "").replace("]", "").replace("-", "minus")
    savefig(Path(out_dir) / f"07_gp_1d_slice_dim{dim_to_vary}_{safe_dim_label}.png")


def plot_gp_slice_2d(
    out_dir,
    model,
    x_ref,
    dim1,
    dim2,
    x1_min,
    x1_max,
    x2_min,
    x2_max,
    n_points=80,
):
    labels = get_feature_labels(model["na"], model["nb"])

    label1 = labels[dim1]
    label2 = labels[dim2]

    x_ref = np.asarray(x_ref).reshape(-1)

    x1_values = np.linspace(x1_min, x1_max, n_points)
    x2_values = np.linspace(x2_min, x2_max, n_points)

    XX1, XX2 = np.meshgrid(x1_values, x2_values)

    X_plot = np.tile(x_ref, (n_points * n_points, 1))
    X_plot[:, dim1] = XX1.reshape(-1)
    X_plot[:, dim2] = XX2.reshape(-1)

    y_pred = predict_from_x_loaded(model, X_plot).reshape(n_points, n_points)

    plt.figure(figsize=(8, 6))
    contour = plt.contourf(XX1, XX2, y_pred, levels=40)
    plt.colorbar(contour, label="Predicted th[k] [rad]")

    plt.xlabel(label1)
    plt.ylabel(label2)
    plt.title(f"2D GP mean slice: {label1} vs {label2}")

    savefig(Path(out_dir) / f"08_gp_2d_slice_dim{dim1}_dim{dim2}.png")

## 11. Generate detailed visualisations for one model

This creates:

1. one-step prediction with uncertainty;
2. free-run simulation;
3. residual time-series plots;
4. residual histograms;
5. predictive standard deviation plot;
6. ARD length-scale plot;
7. 1D posterior slice;
8. 2D posterior mean heatmap.

In [14]:
def make_gp_visualizations(
    model_path,
    project_root=PROJECT_ROOT,
    out_dir=None,
    val_fraction=0.2,
    ref_index=1000,
    dim=-1,
    dim1=-2,
    dim2=-1,
    n_points_1d=250,
    n_points_2d=80,
    n_function_samples=8,
):
    model_path = Path(model_path).resolve()

    if out_dir is None:
        out_dir = model_path.parent / "visualizations"
    else:
        out_dir = Path(out_dir).resolve()

    out_dir.mkdir(parents=True, exist_ok=True)

    print("\nLoading model:")
    print(model_path)

    model = load_gp_model(model_path)

    na = model["na"]
    nb = model["nb"]

    print(f"Loaded GP-NARX model with na={na}, nb={nb}")
    print("Kernel:")
    print(model["gp"].kernel_)

    # Load data and use same validation split.
    data = np.load(TRAIN_DATA_PATH)
    u_all_local = np.asarray(data["u"]).reshape(-1)
    th_all_local = np.asarray(data["th"]).reshape(-1)
    u_all_local = np.clip(u_all_local, -3.0, 3.0)

    _, _, u_val_local, th_val_local = time_series_split(
        u_all_local,
        th_all_local,
        val_fraction=val_fraction,
    )

    X_val, y_true, y_pred, std = one_step_prediction_loaded(
        model,
        u_val_local,
        th_val_local,
        return_std=True,
    )

    print_metrics("One-step prediction", y_true, y_pred)

    plot_one_step_with_uncertainty(out_dir, y_true, y_pred, std)
    plot_residual_time_series(out_dir, y_true, y_pred, name="one_step")
    plot_residual_histogram(out_dir, y_true, y_pred, name="one_step")
    plot_predictive_std(out_dir, std)

    skip = max(na, nb)

    th_sim = simulate_free_run_loaded(
        model,
        u_val_local,
        th_initial=th_val_local,
        skip=skip,
    )

    print_metrics(
        "Free-run simulation",
        th_val_local[skip:],
        th_sim[skip:],
    )

    plot_free_run_simulation(out_dir, th_val_local, th_sim, skip)
    plot_residual_time_series(
        out_dir,
        th_val_local[skip:],
        th_sim[skip:],
        name="free_run_simulation",
    )
    plot_residual_histogram(
        out_dir,
        th_val_local[skip:],
        th_sim[skip:],
        name="free_run_simulation",
    )

    plot_kernel_lengthscales(out_dir, model)

    if ref_index < 0 or ref_index >= len(X_val):
        print(f"ref_index={ref_index} is outside validation range. Using middle sample instead.")
        ref_index = len(X_val) // 2

    x_ref = X_val[ref_index]
    n_features = len(x_ref)

    if dim < 0:
        dim = n_features + dim
    if dim1 < 0:
        dim1 = n_features + dim1
    if dim2 < 0:
        dim2 = n_features + dim2

    labels = get_feature_labels(na, nb)

    print("\nFeature dimensions:")
    for i, label in enumerate(labels):
        print(f"  dim {i}: {label}")

    print(f"\nReference sample index: {ref_index}")
    print(f"1D slice dimension: {dim} ({labels[dim]})")
    print(f"2D slice dimensions: {dim1} ({labels[dim1]}), {dim2} ({labels[dim2]})")

    x_dim_values = X_val[:, dim]
    x_min = np.percentile(x_dim_values, 1)
    x_max = np.percentile(x_dim_values, 99)

    x1_values = X_val[:, dim1]
    x2_values = X_val[:, dim2]

    x1_min = np.percentile(x1_values, 1)
    x1_max = np.percentile(x1_values, 99)
    x2_min = np.percentile(x2_values, 1)
    x2_max = np.percentile(x2_values, 99)

    plot_gp_slice_1d(
        out_dir=out_dir,
        model=model,
        x_ref=x_ref,
        dim_to_vary=dim,
        x_min=x_min,
        x_max=x_max,
        n_points=n_points_1d,
        n_func_samples=n_function_samples,
    )

    plot_gp_slice_2d(
        out_dir=out_dir,
        model=model,
        x_ref=x_ref,
        dim1=dim1,
        dim2=dim2,
        x1_min=x1_min,
        x1_max=x1_max,
        x2_min=x2_min,
        x2_max=x2_max,
        n_points=n_points_2d,
    )

    print("\nDone.")
    print("All plots saved in:", out_dir)

    return out_dir

In [12]:
best_vis_dir = VIS_DIR / f"best_na{int(best_row['na'])}_nb{int(best_row['nb'])}"

make_gp_visualizations(
    model_path=BEST_MODEL_PATH,
    out_dir=best_vis_dir,
    val_fraction=VAL_FRACTION,
    ref_index=1000,
)

NameError: name 'best_row' is not defined

## summary table

In [ ]:
report_table = (
    results_df[
        [
            "order",
            "training_time_s",
            "pred_rmse_rad",
            "pred_rmse_deg",
            "pred_nrms_percent",
            "sim_rmse_rad",
            "sim_rmse_deg",
            "sim_nrms_percent",
        ]
    ]
    .sort_values("sim_rmse_rad")
    .copy()
)

report_table["training_time_s"] = report_table["training_time_s"].round(1)
for col in [
    "pred_rmse_rad",
    "pred_rmse_deg",
    "pred_nrms_percent",
    "sim_rmse_rad",
    "sim_rmse_deg",
    "sim_nrms_percent",
]:
    report_table[col] = report_table[col].round(6)

display(report_table)

print(report_table.to_markdown(index=False))

,order,training_time_s,pred_rmse_rad,pred_rmse_deg,pred_nrms_percent,sim_rmse_rad,sim_rmse_deg,sim_nrms_percent
0,"(5,5)",200.4,0.003541,0.202902,0.684760,0.036455,2.088694,7.048989
4,"(10,5)",216.3,0.003623,0.207573,0.700295,0.042328,2.425222,8.182025
6,"(10,7)",207.5,0.003709,0.212494,0.716894,0.043719,2.504890,8.450803
3,"(10,10)",261.6,0.003834,0.219652,0.741045,0.044018,2.522032,8.508634
7,"(7,10)",216.9,0.003761,0.215489,0.726999,0.044291,2.537681,8.561431
2,"(7,7)",188.1,0.005451,0.312332,1.053924,0.048239,2.763914,9.326480
1,"(6,6)",260.7,0.005883,0.337083,1.137518,0.053025,3.038130,10.252467
5,"(5,10)",181.9,0.006016,0.344711,1.162958,0.058976,3.379060,11.400009


| order   |   training_time_s |   pred_rmse_rad |   pred_rmse_deg |   pred_nrms_percent |   sim_rmse_rad |   sim_rmse_deg |   sim_nrms_percent |
|:--------|------------------:|----------------:|----------------:|--------------------:|---------------:|---------------:|-------------------:|
| (5,5)   |             200.4 |        0.003541 |        0.202902 |            0.68476  |       0.036455 |        2.08869 |            7.04899 |
| (10,5)  |             216.3 |        0.003623 |        0.207573 |            0.700295 |       0.042328 |        2.42522 |            8.18202 |
| (10,7)  |             207.5 |        0.003709 |        0.212494 |            0.716894 |       0.043719 |        2.50489 |            8.4508  |
| (10,10) |             261.6 |        0.003834 |        0.219652 |            0.741045 |       0.044018 |        2.52203 |            8.50863 |
| (7,10)  |             216.9 |        0.003761 |        0.215489 |            0.726999 |       0.044291 |        2.53768 |       